In [3]:

%pip install \
    transformers==4.27.2 \
    datasets==2.11.0 \
    evaluate==0.4.0 \
    rouge_score==0.1.2 \
    loralib==0.1.1 \
    peft==0.3.0

  Using cached transformers-4.27.2-py3-none-any.whl (6.8 MB)
  Using cached datasets-2.11.0-py3-none-any.whl (468 kB)
  Using cached evaluate-0.4.0-py3-none-any.whl (81 kB)
  Using cached rouge_score-0.1.2-py3-none-any.whl
  Using cached loralib-0.1.1-py3-none-any.whl (8.8 kB)
  Using cached peft-0.3.0-py3-none-any.whl (56 kB)
  Using cached regex-2023.10.3-cp39-cp39-win_amd64.whl.metadata (41 kB)
  Using cached tokenizers-0.13.3-cp39-cp39-win_amd64.whl (3.5 MB)
  Using cached dill-0.3.6-py3-none-any.whl (110 kB)
  Using cached responses-0.18.0-py3-none-any.whl (38 kB)
  Using cached absl_py-2.0.0-py3-none-any.whl.metadata (2.3 kB)
  Using cached nltk-3.8.1-py3-none-any.whl (1.5 MB)
INFO: pip is looking at multiple versions of multiprocess to determine which version is compatible with other requirements. This could take a while.
  Using cached multiprocess-0.70.14-py39-none-any.whl (132 kB)
Using cached regex-2023.10.3-cp39-cp39-win_amd64.whl (269 kB)
Using cached absl_py-2.0.0-py3-non

In [1]:
from datasets import load_dataset
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, GenerationConfig, TrainingArguments, Trainer
import torch
import time
import evaluate
import pandas as pd
import numpy as np
from datasets import Dataset

In [2]:
huggingface_dataset_name = "DATA/paleo_filtered3_s.csv"

df = pd.read_csv(huggingface_dataset_name)
huggingface_dataset = Dataset.from_pandas(df)

huggingface_dataset = huggingface_dataset.train_test_split(test_size=0.3)
huggingface_dataset

DatasetDict({
    train: Dataset({
        features: ['author', 'published_at', 'updated_at', 'like_count', 'text', 'video', 'summary'],
        num_rows: 6
    })
    test: Dataset({
        features: ['author', 'published_at', 'updated_at', 'like_count', 'text', 'video', 'summary'],
        num_rows: 3
    })
})

In [3]:
import torch
torch.cuda.is_available()


True

In [4]:
torch.cuda.set_device(0)

In [5]:
#model_name='google/flan-t5-base'
model_name='facebook/bart-large-cnn'
original_model = AutoModelForSeq2SeqLM.from_pretrained(model_name, torch_dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained(model_name)

Some weights of BartForConditionalGeneration were not initialized from the model checkpoint at facebook/bart-large-cnn and are newly initialized: ['model.shared.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [6]:
prefix = "summarize: "


def preprocess_function(examples):
    inputs = [prefix + doc for doc in examples["text"]]
    model_inputs = tokenizer(inputs, max_length=1024, truncation=True)

    labels = tokenizer(text_target=examples["summary"], max_length=128, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [7]:
tokenized_billsum = huggingface_dataset.map(preprocess_function, batched=True)

Map:   0%|          | 0/6 [00:00<?, ? examples/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

In [8]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model_name)

In [9]:
import evaluate

rouge = evaluate.load("rouge")

In [10]:
import numpy as np


def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)

    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in predictions]
    result["gen_len"] = np.mean(prediction_lens)

    return {k: round(v, 4) for k, v in result.items()}

In [11]:
from transformers import AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

Some weights of BartForConditionalGeneration were not initialized from the model checkpoint at facebook/bart-large-cnn and are newly initialized: ['model.shared.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [12]:
training_args = Seq2SeqTrainingArguments(
    output_dir="my_awesome_billsum_model2",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=2,
    predict_with_generate=True,
    fp16=True,
    push_to_hub=True,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_billsum["train"],
    eval_dataset=tokenized_billsum["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

C:\Users\shahe\Anaconda3\envs\torch\lib\site-packages\huggingface_hub\utils\_deprecation.py:127: FutureWarning: 'Repository' (from 'huggingface_hub.repository') is deprecated and will be removed from version '1.0'. Please prefer the http-based alternatives instead. Given its large adoption in legacy code, the complete removal is only planned on next major release.
For more details, please read https://huggingface.co/docs/huggingface_hub/concepts/git_vs_http.
  warnings.warn(warning_message, FutureWarning)
C:\Users\shahe\FYP_MODEL\my_awesome_billsum_model2 is already a clone of https://huggingface.co/SyedShaheer/my_awesome_billsum_model2. Make sure you pull the latest changes with `repo.git_pull()`.
C:\Users\shahe\Anaconda3\envs\torch\lib\site-packages\transformers\optimization.py:391: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable

Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum,Gen Len
1,No log,2.765235,0.401500,0.177700,0.310000,0.310000,68.666700
2,No log,2.516418,0.367500,0.137000,0.307500,0.307500,69.333300


TrainOutput(global_step=2, training_loss=2.8411009311676025, metrics={'train_runtime': 48.0481, 'train_samples_per_second': 0.25, 'train_steps_per_second': 0.042, 'total_flos': 5383900495872.0, 'train_loss': 2.8411009311676025, 'epoch': 2.0})

In [13]:
output_dir = f'./results_check3'

In [14]:
trainer.save_model(output_dir)

Upload file pytorch_model.bin:   0%|          | 1.00/1.51G [00:00<?, ?B/s]

Upload file training_args.bin:   0%|          | 1.00/4.05k [00:00<?, ?B/s]

To https://huggingface.co/SyedShaheer/my_awesome_billsum_model2
   f911a54..0776d74  main -> main

   f911a54..0776d74  main -> main

To https://huggingface.co/SyedShaheer/my_awesome_billsum_model2
   0776d74..9161a3b  main -> main

   0776d74..9161a3b  main -> main



In [8]:
tokenized_datasets = tokenized_datasets.remove_columns(['author', 'published_at', 'updated_at', 'like_count', 'text', 'video', 'summary',])

In [9]:
print(tokenized_datasets)

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 6
    })
    test: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 3
    })
})


In [10]:
output_dir = f'./results_check'

In [11]:
import nltk
import evaluate
nltk.download("punkt", quiet=True)
metric = evaluate.load("rouge")

In [12]:
def compute_metrics(eval_preds):
   preds, labels = eval_preds

   # decode preds and labels
   labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
   decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
   decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

   # rougeLSum expects newline after each sentence
   decoded_preds = ["\n".join(nltk.sent_tokenize(pred.strip())) for pred in decoded_preds]
   decoded_labels = ["\n".join(nltk.sent_tokenize(label.strip())) for label in decoded_labels]

   result = metric.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
  
   return result

In [13]:
L_RATE = 1e-5
BATCH_SIZE = 2
PER_DEVICE_EVAL_BATCH = 2
WEIGHT_DECAY = 0.01
SAVE_TOTAL_LIM = 3
NUM_EPOCHS = 3

# Set up training arguments
training_args = TrainingArguments(
   output_dir="./results_check",
   learning_rate=L_RATE,
   per_device_train_batch_size=BATCH_SIZE,
   weight_decay=WEIGHT_DECAY,
   save_total_limit=SAVE_TOTAL_LIM,
   num_train_epochs=NUM_EPOCHS,
   gradient_accumulation_steps=4,
   fp16=True,
   push_to_hub=False
)




In [14]:
L_RATE = 1e-5
BATCH_SIZE = 1  # Adjusted to fit into 4GB GPU memory
PER_DEVICE_EVAL_BATCH = 2
WEIGHT_DECAY = 0.01
SAVE_TOTAL_LIM = 3
NUM_EPOCHS = 3

# Set up training arguments
training_args = TrainingArguments(
   output_dir="./results_check",
   learning_rate=L_RATE,
   per_device_train_batch_size=BATCH_SIZE,
   weight_decay=WEIGHT_DECAY,
   save_total_limit=SAVE_TOTAL_LIM,
   num_train_epochs=NUM_EPOCHS,
   gradient_accumulation_steps=2,  # Adjusted based on the smaller batch size
   fp16=False,  # Disable mixed precision training
   push_to_hub=False
)


In [21]:
metric = evaluate.load("accuracy")

In [22]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [23]:
# Global Parameters
L_RATE = 1e-5
BATCH_SIZE = 2
PER_DEVICE_EVAL_BATCH = 2
WEIGHT_DECAY = 0.01
SAVE_TOTAL_LIM = 3
NUM_EPOCHS = 3

# Set up training arguments
training_args = TrainingArguments(
   output_dir="./results",
   evaluation_strategy="epoch",
   learning_rate=L_RATE,
   per_device_train_batch_size=BATCH_SIZE,
   per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH,
   weight_decay=WEIGHT_DECAY,
   save_total_limit=SAVE_TOTAL_LIM,
   num_train_epochs=NUM_EPOCHS,
   push_to_hub=False
)

In [24]:
trainer = Trainer(
   model=original_model,
   args=training_args,
   train_dataset=tokenized_datasets["train"],
   eval_dataset=tokenized_datasets["test"],
   tokenizer=tokenizer,
   compute_metrics=compute_metrics
)

In [25]:
trainer.train()

Epoch,Training Loss,Validation Loss


ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 3 dimensions. The detected shape was (2, 3, 512) + inhomogeneous part.

In [17]:
trainer.save_model(output_dir)

In [ ]:
last_checkpoint = "./results_check"

In [ ]:
original_model = AutoModelForSeq2SeqLM.from_pretrained(last_checkpoint, torch_dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained(last_checkpoint)

In [26]:
print(logits.shape)

NameError: name 'logits' is not defined